In [ ]:
import ctypes
import platform

# --- 1. Load the Shared Library ---
# Determine the library file name based on the OS
lib_name = "libsorter.so" if platform.system() != "Windows" else "sorter.dll"

try:
    # Load the library from the current directory
    c_sorter = ctypes.CDLL(f'./{lib_name}')
except OSError as e:
    print(f"Error loading shared library: {e}")
    exit()

# --- 2. Define Function and Callback Signatures ---
# Define the Python type for the C comparison function pointer
CMPFUNC = ctypes.CFUNCTYPE(ctypes.c_int, ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int))

# Define the argument types and return type for our sort_array function
# void sort_array(void* base, size_t num, size_t size, compare_func compar)
c_sorter.sort_array.argtypes = [ctypes.c_void_p, ctypes.c_size_t, ctypes.c_size_t, CMPFUNC]
c_sorter.sort_array.restype = None # 'void' return type

# --- 3. Prepare Data and the Python Callback ---
# The Python list we want to sort
data_py = [5, 2, 8, 1, 9, 4]
print(f"Original Python list: {data_py}")

# Convert the Python list to a C integer array
# This is a crucial step! It allocates C-compatible memory.
CArray = ctypes.c_int * len(data_py)
data_c = CArray(*data_py)

# Python comparison function that will be called by C's qsort
def py_compare_func(a, b):
    """
    Compares two integer pointers. a and b are pointers to c_int,
    so we access their values with a[0] and b[0].
    """
    val_a = a[0]
    val_b = b[0]
    if val_a < val_b:
        return -1
    elif val_a > val_b:
        return 1
    else:
        return 0

# Wrap the Python function in the C-callable type we defined
c_compare_func = CMPFUNC(py_compare_func)

# --- 4. Call the C Function ---
c_sorter.sort_array(
    data_c,                       # The C array
    len(data_c),                  # Number of elements
    ctypes.sizeof(ctypes.c_int),  # Size of one element
    c_compare_func                # The C-callable comparison function
)

# --- 5. View the Result ---
# The original C array object has been sorted in-place.
# We can convert it back to a Python list to see the result.
sorted_py_list = list(data_c)
print(f"Sorted list (from C): {sorted_py_list}")

Original Python list: [5, 2, 8, 1, 9, 4]
Sorted list (from C): [1, 2, 4, 5, 8, 9]
